# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 I am checking: "The Freshness Multiplier" (Finding #4). The
paper says pages older than 365 days that got refreshed saw impressions
jump 57x compared to old pages that weren't touched.

My question: who decided which old pages got refreshed in the first
place? If someone manually picked pages that still mattered like ones
with existing clients or traffic worth saving then the pages chosen
for refresh were probably already better bets before anyone touched
them. The paper even admits the 365+ group is tiny and shaky (283
growing vs just 1 declining page). So I'd want to know: was refreshing
random, or did people pick the pages most likely to bounce back anyway?
If it's the second one, the 57x number might be showing "we're good at
picking winners" more than "refreshing pages actually works."

Finding 2 I am checking: the Logistic Regression model in the ML
appendix that predicts growth vs decline, with 71% accuracy.

My question: how was the train/test data split? Randomly, or grouped
by client? The paper doesn't say. If it's a random split, then the same
client (or even the same page over time) could show up in both the
training data and the test data. That means the model might just be
recognizing patterns it already saw, not actually predicting something
new. This is the exact same problem I had to fix in my own Week-5 model. I switched to a grouped split by client for that reason. The paper
doesn't mention doing that here.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import os, subprocess

REPO_URL = "https://github.com/Shams-Sajid-Rahman/Sajid_FlyRank_AI"
REPO_DIR = "/content/Sajid_FlyRank_AI"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"

print("Connected.")

Connected.


In [2]:
raw = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id, f.report_date,
           f.gsc_impressions, f.gsc_clicks, f.gsc_avg_position,
           d.content_updated_date
    FROM {fact_daily} f
    JOIN {dim_content} d USING (content_hash_id)
    WHERE f.report_date >= '2026-02-01' AND f.report_date < '2026-04-01'
""").df()

raw["report_date"] = pd.to_datetime(raw["report_date"])
end_d = raw["report_date"].max()

raw["is_last30"] = raw["report_date"] > (end_d - pd.Timedelta(days=30))
raw["is_prev30"] = ~raw["is_last30"]

print(raw.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(17196486, 9)


In [3]:
agg = raw.groupby(["client_hash_id", "content_hash_id"]).apply(
    lambda g: pd.Series({
        "imp_prev30": g.loc[g["is_prev30"], "gsc_impressions"].sum(),
        "imp_last30": g.loc[g["is_last30"], "gsc_impressions"].sum(),
        "clk_prev30": g.loc[g["is_prev30"], "gsc_clicks"].sum(),
        "pos_prev30": g.loc[g["is_prev30"], "gsc_avg_position"].mean(),
        "content_updated_date": g["content_updated_date"].iloc[0],
    })
).reset_index()

agg = agg[agg["imp_prev30"] >= 100].copy()
agg["ctr_prev30"] = agg["clk_prev30"] / agg["imp_prev30"]

cutoff_date = end_d - pd.Timedelta(days=30)
agg["content_updated_date"] = pd.to_datetime(agg["content_updated_date"])
agg["days_since_update"] = (cutoff_date - agg["content_updated_date"]).dt.days
agg["days_since_update"] = agg["days_since_update"].clip(lower=0)

agg["is_declining_label"] = (agg["imp_last30"] < 0.8 * agg["imp_prev30"]).astype(int)

print(f"{len(agg):,} content items, decline rate: {agg['is_declining_label'].mean():.3f}")

81,383 content items, decline rate: 0.252


/tmp/ipykernel_4507/4256107055.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg = raw.groupby(["client_hash_id", "content_hash_id"]).apply(


In [4]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

feature_cols = ["imp_prev30", "pos_prev30", "ctr_prev30", "days_since_update"]

def run_model(train_df, test_df):
    X_train, y_train = train_df[feature_cols], train_df["is_declining_label"]
    X_test, y_test = test_df[feature_cols], test_df["is_declining_label"]
    rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    return roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

# --- BEFORE: naive random row-level split (no grouping) ---
train_random, test_random = train_test_split(agg, test_size=0.25, random_state=42)
auc_before = run_model(train_random, test_random)

# --- AFTER: grouped split by client (same as w05) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(agg, groups=agg["client_hash_id"]))
train_grouped = agg.iloc[train_idx]
test_grouped = agg.iloc[test_idx]
auc_after = run_model(train_grouped, test_grouped)

print(f"BEFORE (random split, no grouping):  AUC = {auc_before:.4f}")
print(f"AFTER  (grouped by client):          AUC = {auc_after:.4f}")
print(f"Difference: {auc_before - auc_after:+.4f}")

BEFORE (random split, no grouping):  AUC = 0.6699
AFTER  (grouped by client):          AUC = 0.6125
Difference: +0.0574


Before vs after: when I used a random split (no grouping), the model
looked stronger (0.6690 AUC). Once I switched to a grouped split by
client (same content client never in both train and test), the score
dropped to 0.6125. That's a real gap of about 0.0574.

What this means: some of that extra 0.0574 wasn't the model learning a
real, useful pattern. It was partly the model memorizing quirks of
specific clients it had already seen in training, then getting an easy
win on other pages from those same clients in the "test" set. That's
not a fair test of how the model would do on a brand-new client it's
never seen.

This is the same issue I flagged about the paper's Logistic Regression
model in Section 1. Since the paper doesn't say whether its split was
grouped, its 71% accuracy could be inflated the same way my 0.6690
number was, for the same reason. My grouped-split number, 0.6125, is the
one I trust and the one I'll report going forward.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
print("Leakage audit — checking each feature's timing")
print()
print("Feature            | Source window          | Known before outcome window?")
print("-" * 75)
print("imp_prev30         | prior 30 days           | YES - fully before last30 window")
print("pos_prev30         | prior 30 days           | YES - fully before last30 window")
print("ctr_prev30         | prior 30 days           | YES - fully before last30 window")
print("days_since_update   | at cutoff date          | YES - computed at start of last30")
print()
print("Label (is_declining_label) is built from imp_last30 vs imp_prev30.")
print("None of the four features use imp_last30, clk_last30, or any other")
print("value from the outcome window - only prev30 (input) values are used.")

Leakage audit — checking each feature's timing

Feature            | Source window          | Known before outcome window?
---------------------------------------------------------------------------
imp_prev30         | prior 30 days           | YES - fully before last30 window
pos_prev30         | prior 30 days           | YES - fully before last30 window
ctr_prev30         | prior 30 days           | YES - fully before last30 window
days_since_update   | at cutoff date          | YES - computed at start of last30

Label (is_declining_label) is built from imp_last30 vs imp_prev30.
None of the four features use imp_last30, clk_last30, or any other
value from the outcome window - only prev30 (input) values are used.


In [6]:
# Deliberately add a leaky feature: clk_last30, which comes from the SAME
# window the label is defined on
raw2 = raw.copy()
agg_leak = raw2.groupby(["client_hash_id", "content_hash_id"]).apply(
    lambda g: pd.Series({
        "clk_last30": g.loc[g["is_last30"], "gsc_clicks"].sum(),
    })
).reset_index()

agg_test = agg.merge(agg_leak, on=["client_hash_id", "content_hash_id"])

honest_features = feature_cols
leaky_features = feature_cols + ["clk_last30"]

def quick_score(features, train_df, test_df):
    X_train, y_train = train_df[features], train_df["is_declining_label"]
    X_test, y_test = test_df[features], test_df["is_declining_label"]
    rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    return roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

train_leak = agg_test.iloc[train_idx]
test_leak = agg_test.iloc[test_idx]

honest_auc = quick_score(honest_features, train_leak, test_leak)
leaky_auc = quick_score(leaky_features, train_leak, test_leak)

print(f"Honest features only:        AUC = {honest_auc:.4f}")
print(f"With clk_last30 added in:    AUC = {leaky_auc:.4f}")
print(f"Jump from adding the leak:   {leaky_auc - honest_auc:+.4f}")

/tmp/ipykernel_4507/954318886.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  agg_leak = raw2.groupby(["client_hash_id", "content_hash_id"]).apply(


Honest features only:        AUC = 0.6125
With clk_last30 added in:    AUC = 0.7297
Jump from adding the leak:   +0.1172


I checked my four features one by one: imp_prev30, pos_prev30,
ctr_prev30, and days_since_update. All four only use data from before
the outcome window starts, so none of them can see into the future
the label is trying to predict.

To prove the check actually works, I did the same test as Week 3: I
added a fake "leaky" feature on purpose clk_last30, which comes from
the same 30 days the label itself is built from. The score jumped from
0.613 to 0.730, a big +0.117 gain. That's not the model getting smarter. It's the model reading part of the answer key, since clicks and
impressions in that same window are tied together almost by
definition.

This confirms two things: first, my leakage check is doing its job.
It clearly shows the difference between a fair feature and a leaky one.
Second, my actual model (using only the four honest features) doesn't
have this problem. The real AUC, 0.613, is the honest number.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

The boldest thing I said back in Week 4: "Content that ranks badly and
gets seen by few people is flagged as the top opportunity."

The problem with that: it sounds like a fact, but it was really just a
guess I hadn't tested yet. When I actually checked it in Weeks 5 and 6,
the real numbers were a lot weaker. My honest model only hits 0.613
AUC barely better than flipping a coin and my original Week 4 rule
scored basically random (0.512) once I properly tested it.

Here's the safer way to say it: "Pages with worse position and lower
past impressions do tend to get flagged more often by my rule but
it's a weak, measured pattern (AUC 0.613), not a strong or reliable
predictor. It's something worth a human looking at, not something to
trust blindly."

Same basic idea, but now it's honest about how shaky the signal
actually is, and it makes clear this is meant to help a person decide,
not replace their judgment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.